In [79]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
import yaml
from pathlib import Path

# Fleet model calibration notebook

The calibration is provided for year 2019. Then the total asks per aircraft type are estimated given the fleet at the end of 2024.

1. Folder: `fleet_data_sources`
This folder contains the **source data** required for the calibration processes, including:

- List of modeled aircraft ICAO codes. 
- **BTS** traffic data for 2010, 2019 and 2025. It allows to cover a variety of aircraft which may be absent today of the US market, while still avoiding to manipulate a very heavy dataset. Identified with aircraft ICAO codes.
- **Seymour's simplified fuel model coefficients**.
- **Private TAA fleet data file**, not public. Every commercial passenger aircraft (no freight), identified with aircraft ICAO codes, with more than 10 active aircraft in 2019 or still produced.
- **Airport datase**, provided by BTS as support table (@mention ISA). It is used to apply weights on the assignment (avoiding over representation of international flights in regional datasets). 

2. Folder: `fleet_calibrated_inputs_processed_pviry_repository`
This folder contains the **processed input data** using various functions and codes developed during **Paco Viry's thesis**. It includes:

- Quantitative mesure of the **impact of route distance on aircraft utilization**.
- Coefficient capturing the **impact of aircraft age on utilization** : -x% per year.
- Sets of coefficients capturing **type-specific and age-related retirement propensity**.

3. Folder: `fleet_calibrated_inputs_processed_here`
This folder contains the  **processed input data** built upon `data_sources` and `calibrated_inputs_processed_pviry_repository` in the notebook.
- Fleet structure in 2019 per aircraft ICAO code and built year.
- Average number of seats per aircraft ICAO code (from BTS). When not available in BTS, it was completed given aircraft public information.
- E/ASK calculations per ICAO code.
- Productivity for age 0 aircraft (ASK/ac/y) calculated per ICAO code. When not available, it was copied on a comparable aircraft. Total estimated ASKs per ICAO type in 2019. (including a correction factor to match the total ASKs calculated through the fleet structure, seats capacities and productivity mechanisms and ICAO total market ASKs in 2019).

## Fleet structure calculations

In [80]:
# def date_to_float_year(date):
#     if type(date)!= float :
#         start_of_year = pd.Timestamp(year=date.year, month=1, day=1)
#         start_of_next_year = pd.Timestamp(year=date.year + 1, month=1, day=1)
#         year_length = (start_of_next_year - start_of_year).days
#         days_since_start_of_year = (date - start_of_year).days
#         return date.year + days_since_start_of_year / year_length
#     else :
#         return date
    
# fleet_ini = pd.read_excel('fleet_data_sources/fleet_extract_250610.xlsx')
# fleet_ini = fleet_ini[fleet_ini['Airframe Body Type'].isin(['Narrowbody', 'Small Widebody', 'Turboprop', 'Regional Jet', 
#                                                             'Medium Widebody', 'Large Widebody'])]
# etatic_operators = ['USAF','US NAVY','NATO','UZBEKISTAN A.F.', 'BDS USAF TANKER', 'GERMANY A.F.','USAF',  'RUSSIA A.F.','JAPAN GOVT', 'MALI GOVT', 'LIBYA A.F.', 'SPAIN A.F.', 'SUDAN A.F.',
#        'ANGOLA GOVT', 'TURKEY GOVT', 'FRANCE A.F.','ISRAEL A.F.', 'GUINEA EQUATORIAL GOVT', 'UKRAINE A.F.', 'UKRAINE GOVT',
#         'JAPAN AIR SELF-DEFENSE FORCE','SRI LANKA A.F.', 'BANGLADESH GOVT',  'IRAQ GOVT',  'TURKMENISTAN AIRLINES',
#        'PERU NAVY', 'TURKEY A.F.', 'EGYPT A.F.','INDIA A.F.', 'PAKISTAN A.F.','CHINA A.F.', 'ROYAL AIR FORCE', 
#         'ALGERIA GOVT', 'RUSSIA GOVT',  'CHINA GOVT', 'MEXICO A.F.', 'AFGHANISTAN A.F.', 'CROATIA A.F.', 'CONGO A.F.', 'PERU A.F.', 'CONGO GOVT', 
#        'MALAYSIA A.F.', 'ARGENTINA ARMY',  'AUSTRALIA A.F.', 'ANGOLA NAVY','SYRIAN ARAB A.F.', 'COLOMBIA A.F.', 'YEMEN A.F.', 'SOUTH KOREA GOVT', 'ROMANIA A.F.', 
#        'UNITED NATIONS', 'US COAST GUARD', 'INDONESIA NAVY', 'THAILAND ROYAL POLICE', 'URUGUAYA A.F.', 'IRAN GOVT', 'JORDAN A.F.', 'US ARMY', 'CHAD GOVT',
#         'INDONESIA A.F.', 'BELARUS GOVT', 'CAMEROON A.F.', 'US GOVT DEPT OF STATE', 'EQUATORIAL GUINEA GOVT',
#        'PORTUGAL A.F.','GUINEA-BISSAU GOVT', 'ZIMBABWE A.F.', 'VIETNAM GOVT', 'CONGO BRAZAVILLE',  'BLACKWATER AVIATION',
#        'SUDANESE STATES AVIATION', 'SPAIN GOVT', 'MALTA GOVT', 'TOGO A.F.', 'SOUTH AFRICA A.F.',
#        'THAILAND GOVT','ECUADOR ARMY','SUDAN GOVT', 'MOROCCO GOVT', 'CHILE A.F.', 'ECUADOR NAVY', 'PANAMA A.F.', 'CHILE ARMY', 'SENEGAL A.F.',
#         'BULGARIA A.F.',  'IRAN A.F.','COLOMBIA GOVT','THAILAND NAVY',  'BOTSWANA GOVT', 'CZECH GOVT', 
#         'LITHUANIA GOVT','ITALY A.F.', 'AZERBAIJAN GOVT', 'LIBYA GOVT', 'GREECE A.F.','RUSSIAN FEDERAL SPACE AGENCY', 'INDONESIA GOVT.', 'KAZAKHSTAN GOVT', 'SERBIA GOVT',
#        'SWEDEN COAST GUARD', 'INDONESIA ARMY', 'PERU GOVT',  'TRANSKEI GOVT', 'ARMENIAN GOVT', 
#         'PARAGUAY A.F.', 'BELGIUM A.F.', 'MALAWI GOVT', 'DOMINICAN REPUBLIC GOVT', 'LESOTHO GOVT', 'MEXICO GOVT', 
#         'UAE A.F.', 'VENEZUELA NAVY', 'BRUNEI GOVT', 'THAILAND ARMY','ZAIRE A.F.', 'SOUTH AFRICAN GOVT', 'IRELAND COAST GUARD',
#        'UAE GOVT', 'US GOVT DEA', 'BELORUSSIA A.F.', 'OMAN POLICE','PAPUA NEW GUINEA GOVT','US GOVT DEPT OF JUSTICE', 'GABON A.F.',
#        'ARGENTINA NAVY', 'CHILE NAVY', 'SAUDI ARABIA GOVT', 'ETHIOPIA A.F.']
# fleet_ini = fleet_ini[~fleet_ini['Current Operator'].isin(etatic_operators)]
# fleet_ini = fleet_ini[['Type', 'Model', 'Exp. Delivery', 'Status', 'Date', 'Year Built']] 


# taa_icao_names = pd.read_csv('fleet_data_sources/aircraft_codes_taa_icao.csv', sep=';')
# dict_taa_icao = dict(zip(taa_icao_names['Aircraft Type'], taa_icao_names['ICAO_Code']))
# fleet_ini['Aircraft Type'] = fleet_ini['Type'].astype(str) + fleet_ini['Model'].astype(str)
# fleet_ini['Aircraft Type'] = fleet_ini['Aircraft Type'].map(dict_taa_icao)
# fleet_ini = fleet_ini[~fleet_ini['Aircraft Type'].isna()]

# fleet_ini['Year Built']= fleet_ini['Year Built'].fillna(fleet_ini['Exp. Delivery'])
# fleet_ini = fleet_ini[fleet_ini['Year Built']!='12/30/1899']
# fleet_ini = fleet_ini[~fleet_ini['Year Built'].isna()]
# fleet_ini['Year Built'] = fleet_ini['Year Built'].apply(date_to_float_year)
# fleet_ini['Date'] = fleet_ini['Date'].apply(date_to_float_year)

# fleet_ini = fleet_ini[['Aircraft Type','Year Built', 'Status', 'Date']]
# fleet = fleet_ini[fleet_ini['Status'].isin(['Active', 'Stored', 'Written Off'])].reset_index(drop=True)
# fleet = fleet[~fleet['Aircraft Type'].isin(['AN12','DC95', 'DC92', 'L101', 'B742', 'DC91', 'AN32','IL86',
#        'L188', 'B743', 'DC10', 'MD81', 'DC93', 'B74S','B703'])] #less than 10 active aircraft in 2019 - and none in perspective
# fleet.to_csv('fleet_data_sources/fleet_data.csv')

In [96]:
fleet = pd.read_csv('fleet_data_sources/fleet_data.csv')
fleet.drop(columns=['Unnamed: 0'], inplace = True)
fleet['Year Built'] = fleet['Year Built'].astype(int)
y_min = fleet['Year Built'].min()

In [97]:
#'A338','B3XM' are present in 2024 but not 2019 we need to calibrate the representing fleet in 2024 then use its columns
#used to estimate traffic at end 2024/beginning 2025
obs_year = 2024
fleet_selec = fleet[
        (fleet['Year Built'] <= obs_year) & (fleet['Year Built'] > 1979) &
        ((fleet['Status'].isin(['Active', 'Stored'])) | (fleet['Date'] > obs_year+1))
    ] 

min_year = fleet_selec['Year Built'].min()
max_year = obs_year
all_years = range(min_year, max_year + 1)
counts = fleet_selec.groupby(['Aircraft Type', 'Year Built']).size().unstack(fill_value=0).reindex(columns=all_years, fill_value=0) #fleet content january 2019
counts.to_excel('fleet_calibrated_inputs_processed_here/agg_fleet_end_'+str(obs_year)+'.xlsx')

#used to estimate real productivity.
obs_year = 2019
fleet_selec1 = fleet[
    (fleet['Year Built'] <= obs_year-1) & (fleet['Year Built'] > 1979) &
    ((fleet['Status'].isin(['Active', 'Stored'])) | (fleet['Date'] > obs_year))
] 


min_year = fleet_selec1['Year Built'].min()
max_year = obs_year
all_years = range(min_year, max_year + 1)
counts1 = fleet_selec1.groupby(['Aircraft Type', 'Year Built']).size().unstack(fill_value=0).reindex(columns=all_years, fill_value=0) #fleet content january 2019


fleet_selec2 = fleet[
    (fleet['Year Built'] <= obs_year) & (fleet['Year Built'] > 1979) &
    ((fleet['Status'].isin(['Active', 'Stored'])) | (fleet['Date'] > obs_year + 1)) #fleet content january 2020
]

counts2 = fleet_selec2.groupby(['Aircraft Type', 'Year Built']).size().unstack(fill_value=0).reindex(columns=all_years, fill_value=0)
counts1 = counts1.reindex(index=counts.index, fill_value=0)
counts2 = counts2.reindex(index=counts.index, fill_value=0)
counts_f = (counts1 + counts2) / 2 #average composition during the year.
counts_f.to_excel('fleet_calibrated_inputs_processed_here/agg_fleet_avg_'+str(obs_year)+'.xlsx')

list_prov_taa = sorted(list(counts.index))   

## Energy per ASK calculations

### Utilisation, seats and productivity estimations (BTS only)

In [98]:
# traffic_bts = pd.read_csv('fleet_data_sources/T_T100_SEGMENT_ALL_CARRIER_2010.csv')
# df = pd.read_csv('fleet_data_sources/T_T100_SEGMENT_ALL_CARRIER_2019.csv')
# traffic_bts = pd.concat([traffic_bts, df], ignore_index=True)
# df = pd.read_csv('fleet_data_sources/T_T100_SEGMENT_ALL_CARRIER_2025.csv')
# traffic_bts = pd.concat([traffic_bts, df], ignore_index=True)
# traffic_bts = traffic_bts[(traffic_bts['DEPARTURES_PERFORMED']>0)&(traffic_bts['SEATS']>0)&(traffic_bts['DISTANCE']>0)]

# bts_icao = pd.read_csv('fleet_data_sources/aircraft_codes_icao_joined.csv', sep=';')
# bts_icao_dict = dict(zip(bts_icao['Code'], bts_icao['ICAO_Code']))

# traffic_bts['Aircraft Type'] = traffic_bts['AIRCRAFT_TYPE'].map(bts_icao_dict)
# traffic_bts = traffic_bts[~traffic_bts['Aircraft Type'].isna()]
# traffic_bts = traffic_bts[['DEPARTURES_PERFORMED','SEATS','DISTANCE','ORIGIN_AIRPORT_ID','DEST_AIRPORT_ID','YEAR','Aircraft Type']]
# traffic_bts.to_csv('fleet_data_sources/traffic_data_bts_2010_2019_2025.csv')

In [99]:
traffic_bts = pd.read_csv('fleet_data_sources/traffic_data_bts_2010_2019_2025.csv')
traffic_bts.drop(columns=['Unnamed: 0'], inplace = True)
# weighting the connections
airports = pd.read_csv('fleet_data_sources/bts_airports.csv')[['AIRPORT_ID','AIRPORT_COUNTRY_NAME']]
us_airports = airports[airports['AIRPORT_COUNTRY_NAME']=='United States']
us_id = us_airports['AIRPORT_ID'].unique() #some inconsistencies regarding porto rico (minor)

traffic_bts['weight'] = 0
traffic_bts.loc[traffic_bts['ORIGIN_AIRPORT_ID'].isin(us_id), 'weight']+=0.5
traffic_bts.loc[traffic_bts['DEST_AIRPORT_ID'].isin(us_id), 'weight']+=0.5

C:\Users\p.viry\AppData\Local\Temp\ipykernel_804\1928288757.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.5 0.5 0.5 ... 0.5 0.5 0.5]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  traffic_bts.loc[traffic_bts['ORIGIN_AIRPORT_ID'].isin(us_id), 'weight']+=0.5


In [100]:
list_prov_bts = traffic_bts['Aircraft Type'].unique()
icao_taa_in_bts = [a for a in list_prov_taa if a in list_prov_bts] #based on bts.
icao_taa_not_in_bts = [a for a in list_prov_taa if a not in list_prov_bts] #extrapolated.

In [101]:
traffic_bts_s = traffic_bts[traffic_bts['Aircraft Type'].isin(icao_taa_in_bts)]

# ground time dependence on flight time.
gt_est = pd.read_excel('fleet_calibrated_inputs_processed_pviry_repository/GT_contr_estimates_EC_preCOVID.xlsx')
gt_est['abscisse'] = np.log(gt_est['abscisse'])
FT_appro_lin = np.array(gt_est['abscisse'])
GT_appro_lin = np.array(gt_est['GT_contr'])
linear_interp = interp1d(
        FT_appro_lin,
        GT_appro_lin,
        kind='linear', bounds_error=False,fill_value=(GT_appro_lin[0], GT_appro_lin[-1]))

# flight time estimation. As a simplification, it is not type-specific but distance-specific
#it is supposed that cruise speed is v1 at lower ranges, then v2
#a 0.5 supplement climb delay is continuously added through an exponential including a climb speed vc (limit = supp, derivative at 0 = vc)
# could be improved if statistically based on eurocontrol data.
d_seuil=5000/1.852 #medium/long distance threshold nm
v_1=830/1.852 #short medium distance cruise speed nm/h
v_2=900/1.852 #long distance cruise speed nm/h
v_c = 500/1.852 #climb/descent speed nm/h
supp=0.5 #asymptotic additionnal time accounting for climb and descent, hours
traffic_bts_s['Estimated_FT'] = (((traffic_bts_s['DISTANCE']<d_seuil)*traffic_bts_s['DISTANCE']/v_1 +
                          (traffic_bts_s['DISTANCE'] >= d_seuil) *traffic_bts_s['DISTANCE'] / v_2)
                          + supp*(1-np.exp(-traffic_bts_s['DISTANCE']/(v_c*v_1)*(v_1-v_c)/supp)))
gt_cont = linear_interp(np.log(traffic_bts_s['Estimated_FT'].values+0.3)) # 0.3 is needed to include taxi in/out (not in the estimates used)
traffic_bts_s['time_immobilisation'] = traffic_bts_s['DEPARTURES_PERFORMED']*(traffic_bts_s['Estimated_FT'] + gt_cont) #total time needed to operate the connection


traffic_bts_s['Distance_tot_km'] = 1.852*traffic_bts_s['DEPARTURES_PERFORMED']*traffic_bts_s['DISTANCE']
traffic_bts_s['ASK'] = 1.852*traffic_bts_s['SEATS']*traffic_bts_s['DISTANCE']

traffic_bts_s_group = traffic_bts_s[['Aircraft Type', 'ASK', 'Distance_tot_km','time_immobilisation']].groupby('Aircraft Type').sum().reset_index()
traffic_bts_s_group['yearly_distance_km'] = traffic_bts_s_group['Distance_tot_km']/traffic_bts_s_group['time_immobilisation']*(365.25-11.5)*24 #11.5 days of repair per year scholz
traffic_bts_s_group['avg_seats'] = traffic_bts_s_group['ASK']/traffic_bts_s_group['Distance_tot_km']


# computed utilisation and seats data
yearly_dist_dict = dict(zip(traffic_bts_s_group['Aircraft Type'],traffic_bts_s_group['yearly_distance_km']))
avg_seats_dict = dict(zip(traffic_bts_s_group['Aircraft Type'],traffic_bts_s_group['avg_seats']))
avg_seats_dict['SH33'] = avg_seats_dict['SH36']-3

C:\Users\p.viry\AppData\Local\Temp\ipykernel_804\1167513053.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  traffic_bts_s['Estimated_FT'] = (((traffic_bts_s['DISTANCE']<d_seuil)*traffic_bts_s['DISTANCE']/v_1 +
C:\Users\p.viry\AppData\Local\Temp\ipykernel_804\1167513053.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  traffic_bts_s['time_immobilisation'] = traffic_bts_s['DEPARTURES_PERFORMED']*(traffic_bts_s['Estimated_FT'] + gt_cont) #total time needed to operate the connection
C:\Users\p.viry\AppD

In [102]:
# non-available data.
#operation caracteristics will be associated to a "similar aircraft" present in bts.
absent_types_equiv = {'F70':'E170','E195':'E190','F50':'AT43','E75S':'E75L','AT76':'AT72','AT46':'DHC6','AT75':'AT72','AT45':'DHC6','MD83':'B733','F100':'E190', 'F28':'CRJ2',
                      'CRJX':'CRJ9', 'BCS1':'A319', 'MD82':'B732', 'B77L':'B77W', 'BCS3':'A320','A338':'A339','B3XM':'B39M',
                 'B461':'B712', 'B703':'B722','B773':'B77W', 'AN24':'DH8A', 'D328':'AT43', 'YK40':'CRJ2', 'B463':'B712', 'DHC7':'DHC6', 'F27':'DH8B',
             'B462':'B712', 'YK42':'CRJ7', 'E110':'B190', 'AJ27':'CRJ2', 'MD88':'MD87', 'IL96':'A332', 'JS32':'E120', 'T154':'B734', 'B721':'B722', 
             'IL62':'B763', 'JS41':'E120', 'A19N':'A319', 'IL18':'AT72', 'T134':'CRJ2', 'C919':'A320'}

#seats caracteristics are hand based on typical configurations. These numbers could be adjusted.
absent_types_seats = {'F70':75,'E195':115,'F50':45,'E75S': 80,'AT76':70,'AT46':45,'AT75':70,'AT45':42,'MD83':150,'F100': 100, 'F28': 80,
                      'CRJX':100, 'BCS1':110, 'MD82':150, 'B77L':350, 'BCS3':140,'A338':245,'B3XM':190,
                 'B461':80, 'B703':130,'B773':350, 'AN24':52, 'D328':30, 'YK40': 30, 'B463':100, 'DHC7':50, 'F27':60,
             'B462':90, 'YK42': 100, 'E110':20, 'AJ27': 80, 'MD88':140, 'IL96':260, 'JS32': 15, 'T154': 130, 'B721': 140, 
             'IL62': 160, 'JS41':29, 'A19N': 140, 'IL18':120, 'T134':80, 'C919':150}

#productivity dictionnary
productivity_dict = dict()
for ac_new in list_prov_taa:
    if ac_new in icao_taa_not_in_bts:
        ac_old = absent_types_equiv[ac_new]
        yearly_dist_dict[ac_new] = yearly_dist_dict[ac_old]
        avg_seats_dict[ac_new] = absent_types_seats[ac_new]
        productivity_dict[ac_new] = yearly_dist_dict[ac_new] * avg_seats_dict[ac_new]
    else :
        productivity_dict[ac_new] = yearly_dist_dict[ac_new] * avg_seats_dict[ac_new]

### Energy efficiency calculations

In [103]:
fb_coefficients = pd.read_csv('fleet_data_sources/ac_model_coefficients_seymmour.csv')
list_prov_seymm = fb_coefficients['ac_code_icao'].unique()
icao_taa_in_seymm = [a for a in list_prov_taa if a in list_prov_seymm] #based on bts.
icao_taa_not_in_seymm = [a for a in list_prov_taa if a not in list_prov_seymm] #extrapolated.

#fuel consumption is based on seymmour when available, and if not available is based on close aircraft similar in seymmour. Here is a correspondance dictionnary. 
# It is sometimes very unprecise, but in this cases the associated aircraft are in very low volume.
absent_types_equiv_seymm = {'AT76':'AT72', 'AT46':'AT43', 'AT75':'AT72', 'AT45':'AT43','A338':'A339','B3XM':'B39M',
 'B703':'B762', 'SH36':'AT43', 'B721':'B722', 'IL62':'B762', 'A19N':'A20N', 'IL18':'B722', 'SH33':'AT43', 'C919':'A320'}

In [104]:
fuel_efficiency_dict = dict()
for ac_type in list_prov_taa :
    if ac_type in icao_taa_not_in_seymm:
        ac_type_fb = absent_types_equiv_seymm[ac_type]
    else :
        ac_type_fb = ac_type
    if ac_type in icao_taa_not_in_bts:
        ac_type_ut = absent_types_equiv[ac_type]
    else :
        ac_type_ut = ac_type
        
    coeffs = (fb_coefficients[fb_coefficients['ac_code_icao']==ac_type_fb][['reduced_fuel_intercept','reduced_fuel_a1','reduced_fuel_a2']].values)[0]
    utilisation = traffic_bts_s[traffic_bts_s['Aircraft Type']==ac_type_ut][['DISTANCE', 'DEPARTURES_PERFORMED']].values
    fb = coeffs[0]+coeffs[1]*(utilisation[:,0])**2+coeffs[2]*utilisation[:,0]
    e_ask = (fb*utilisation[:,1]).sum()/(utilisation[:,0]*utilisation[:,1]).sum()/avg_seats_dict[ac_type]
    fuel_efficiency_dict[ac_type] = 46.4*e_ask
    
aircraft_types_df = pd.DataFrame({'average_seats': avg_seats_dict, 'distance_aircraft_year(km)': yearly_dist_dict, 
                                  'ask_aircraft_year_age0': productivity_dict, 'MJ_fuel_ask': fuel_efficiency_dict}).sort_index(ascending=True)
aircraft_types_df['Aircraft Type'] = aircraft_types_df.index

In [105]:
aircraft_types_df.to_excel('fleet_calibrated_inputs_processed_here/aircraft_type_key_parameters.xlsx')

## Total ASKs corrections

Based on the productivity estimates, and impact of age on utilisation, this section deduces total AKSs in 2019.
This total quantity is compared to an official value. Because it was calculated based on operationnal capabilities and aging coefficients, it is no surprise that it produces more ASKs than the real worls. Indeed, other inactivity factors such as the PW engine crisis could not be anticipated. Furthermore, here the assumption is made that aircraft speed depends on the range which is true, but does not consider the type dependency, in particular regarding turboprops aircraft. Regarding these, it should however be compensated by their shorter average tunraround time. This results in a corrective factor used to adjust the real productivity per aircraft.

In [106]:
aircraft_types_df = pd.read_excel('fleet_calibrated_inputs_processed_here/aircraft_type_key_parameters.xlsx')
aircraft_types_df.drop(columns =['Unnamed: 0'], inplace = True)
aircraft_types_df = aircraft_types_df[['Aircraft Type', 'average_seats', 'distance_aircraft_year(km)', 'ask_aircraft_year_age0','MJ_fuel_ask']]
fleet_2019 = pd.read_excel('fleet_calibrated_inputs_processed_here/agg_fleet_avg_2019.xlsx')
fleet_2024 = pd.read_excel('fleet_calibrated_inputs_processed_here/agg_fleet_end_2024.xlsx')

age_utilisation_sensibility = -0.01 # /year, relative utilisation decrease for one year older.

In [107]:
fleet_volumes_2019 = fleet_2019.values[:,1:]
fleet_volumes_2024 = fleet_2024.values[:,1:]
productivity_types = aircraft_types_df['ask_aircraft_year_age0'].values


ages_2019 = (fleet_volumes_2019.shape[1]-1- np.arange(fleet_volumes_2019.shape[1]))
activity_indicator_2019 = np.exp(age_utilisation_sensibility*ages_2019)
ask_arrays_2019 = fleet_volumes_2019 * activity_indicator_2019[None, :]*productivity_types[:, None]
ask_types_2019 = ask_arrays_2019.sum(axis =1)

ages_2024 = (fleet_volumes_2024.shape[1]-1- np.arange(fleet_volumes_2024.shape[1]))
activity_indicator_2024 = np.exp(age_utilisation_sensibility*ages_2024)
ask_arrays_2024 = fleet_volumes_2024 * activity_indicator_2024[None, :]*productivity_types[:, None]
ask_types_2024 = ask_arrays_2024.sum(axis =1)

In [108]:
reference_rpk_2019 = 8.686e12 #source icao
reference_lf_2019 = 0.81 #source iata
predicted_rpk = reference_lf_2019*ask_arrays_2019.sum()
print(str(predicted_rpk/1e12)+' e12 RPKs predicted for 2019')
correction_factor = reference_rpk_2019/predicted_rpk
print('+ '+str(int(100*(1/correction_factor-1)+0.5))+'% overestimation -> '+str(int(100*(correction_factor-1)+0.5))+'% utilisation correction')

11.363473798787664 e12 RPKs predicted for 2019
+ 31% overestimation -> -23% utilisation correction


In [109]:
aircraft_types_df['adj_distance_aircraft_year(km)'] = aircraft_types_df['distance_aircraft_year(km)']*correction_factor
aircraft_types_df['adj_ask_aircraft_year_age0'] = aircraft_types_df['ask_aircraft_year_age0']*correction_factor
aircraft_types_df['total_ask_produced_2024'] = ask_types_2024*correction_factor

In [111]:
retirement_propensions = pd.read_excel('fleet_calibrated_inputs_processed_pviry_repository/TAA_propension_lin_icao_am.xlsx')
age_retirement_sensibility = retirement_propensions['Beta'].values[0]
aircraft_types_df.loc[aircraft_types_df['Aircraft Type'].isin(retirement_propensions.columns[2:]),'retirement_propensions'] = retirement_propensions.values[0,2:]
aircraft_types_df.fillna(0, inplace = True)

In [112]:
aircraft_types_df.to_excel('fleet_calibrated_inputs_processed_here/aircraft_type_key_parameters.xlsx')

In [113]:
other_parameters = pd.DataFrame({'age_utilisation_sensibility' : [age_utilisation_sensibility], 'age_retirement_sensibility':[age_retirement_sensibility]})
other_parameters.to_excel('fleet_calibrated_inputs_processed_here/other_parameters.xlsx')

In [115]:
data =list(aircraft_types_df['Aircraft Type'])
with open("fleet_calibrated_inputs_processed_here/default_aircraft_classification_empty.yaml", "w") as f:
    yaml.dump(
        data,
        f,
        sort_keys=True,
        default_flow_style=False
    )